# Data merging
- Mergozzo
- 1.0.0

## Context

   ![Pipeline](images/Pipeline.png)

## Project creation and configuration
- *JAVA_HOME* must be set for PySpark to be imported correctly.
- *spark.sql.shuffle.partitions* sets the default number of partitions Spark creates when it redistributes data. For example during joins, grouping, aggregations, sorting or distinct etc.

In [ ]:
import os
from pyspark.sql import *
from delta import *

if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

builder = (
    SparkSession
        .builder
        .appName("Mergozzo")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

session = configure_spark_with_delta_pip(builder).getOrCreate()
session.conf.set("spark.sql.shuffle.partitions", "20")
session.conf.set("spark.sql.debug.maxToStringFields", "100")

## Datastore creation and configuration
With creation of database **alpinism** the datastore **spark-warehouse** will be automatically created too.

In [ ]:
session.sql("DROP DATABASE IF EXISTS alpinism CASCADE")
session.sql("CREATE DATABASE IF NOT EXISTS alpinism")
session.sql("USE alpinism")

## Data merge analysis
1. General rules for clause frequencies and conditions:

    - A `MERGE` operation can contain multiple `whenMatched` and `whenNotMatched` clauses.
    - Each individual clause can specify only **one action** e.g. a single clause cannot perform both an `UPDATE` and a `DELETE`.
    - Multiple `whenMatched` clauses with `UPDATE` or `DELETE` actions are allowed. They are evaluated sequentially in the order specified.
    - All `whenMatched` clauses, except the last one, must have an explicit condition.
    - Modern systems supports multiple `whenNotMatched` clauses as well as `whenNotMatchedBySource` clauses with conditions.

---
2. The `whenMatched` clauses are executed when a source row matches a target row based on the merge condition. They have the following semantics:

    - Each `whenMatched` clause can contain at most one Update or one Delete action. The Update action updates only the specified columns of the matched target row, while the Delete action deletes the matched row.
    - Each `whenMatched` clause can have an optional condition. If specified, the `UPDATE` or `DELETE` action is executed for a matching source-target row pair only when that condition evaluates to `true`.
    - Multiple `whenMatched` clauses are evaluated in the order specified. All `whenMatched` clauses, except the last one, must have explicit conditions.
    - If none of the `whenMatched` conditions evaluate to `true` for a matching row pair, the target row is left unchanged.
    - To update all columns of the target table with the corresponding columns of the source dataset, use `whenMatched(...).updateAll()`.

   ![WhenMatchedDelete](images/whenMatchedDelete.png)

3. The `whenNotMatched` clauses are executed when a source row does not match any target row based on the merge condition. The `whenNotMatched` clauses have the following semantics:

    - Each `whenNotMatched` clause can only have the `INSERT` action. The new row is generated based on the specified columns and their corresponding expressions. It is not required to specify all the columns in the target table; for unspecified target columns, `NULL` is inserted.
    - Each `whenNotMatched` clause can have an optional condition. If the clause condition is present, the source row is inserted only when the condition is satisfied for that row. Otherwise, the source row is ignored.
    - If there are multiple `whenNotMatched` clauses, they are executed in the order they are specified. All `whenNotMatched` clauses, except the last one, must have conditions.
    - To insert all columns of the target Delta Lake table using the corresponding columns of the source dataset, use `whenNotMatched(...).insertAll()`.

   ![WhenNotMatchedInsertAll](images/whenNotMatchedInsertAll.png)

4. The `whenNotMatchedBySource` clauses are executed when a target row does not match any source row based on the merge condition. They have the following semantics:

    - Each `whenNotMatchedBySource` clause can contain at most one `DELETE` or `UPDATE` action.
    - Each `whenNotMatchedBySource` clause can have an optional condition. If present, the target row is deleted or updated only when the condition is satisfied. Otherwise, the target row is left unchanged.
    - If there are multiple `whenNotMatchedBySource` clauses, they are evaluated in the order they are specified. All clauses, except the last one, must have conditions.
    - By definition, `whenNotMatchedBySource` clauses do not have a corresponding source row, so **source columns cannot be referenced**. However, for each updated target column, you can specify a literal value or an expression based on existing target columns e.g. `SET target.del_count = target.del_count + 1` or `SET is_active = false`.

   ![WhenNotMatchedBySourceDelete](images/whenNotMatchedBySourceDelete.png)


### Table schema creation and data ingestion
#### Dataset ⟶ The **alpinist** table stores the first mountaineers, both men and women, to reach the summit of K2

In [ ]:
data = [
    (1, "Achille", "Compagnoni", "1914-09-26", "Italian"),
    (2, "Ardito", "Desio", "1897-04-18", "Italian"),
    (3, "Lino", "Lacedelli", "1925-12-04", "Italian"),
    (4, "Wanda", "Rutkiewicz", "1944-12-13", "Polish")
]
df_alpinist = session.createDataFrame(data).toDF("id", "first_name", "last_name" , "birthdate", "nationality").repartition(1)

df_alpinist.write.format("delta").saveAsTable("alpinist")
session.table("alpinist").show()

alpinist = DeltaTable.forName(session, "alpinist")
alpinist.toDF().show()

### Analysis 1 ⟶ Clause **whenNotMatchedInsertAll**
#### Dataset A ⟶ Dataset of Nepalese mountaineers [Delta Lake API & PySpark API]

In [ ]:
data_nepalese = [
    (5, 'Nirmal', 'Purja', '2003-07-25', 'Nepalese'),
    (6, 'Kamil', 'Rita', '1970-01-17', 'Nepalese')
]
df_nepalese = session.createDataFrame(data_nepalese).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_nepalese.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

alpinist.toDF().show()

#### Dataset B ⟶ Dataset of French mountaineers [PySpark API]

In [ ]:
data_french = [
    (7, 'Lionel', 'Terray', '1921-07-25', 'French'),
    (8, 'Maurice', 'Herzog', '1919-01-15', 'French')
]
session.createDataFrame(data_french).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1).createOrReplaceTempView("alpinist_french")

session.sql("""
    MERGE INTO alpinist AS target
    USING alpinist_french AS source
    ON target.id = source.id
    WHEN NOT MATCHED THEN
      INSERT *
""")

alpinist.toDF().show()

### Analysis 2 ⟶ Clause **whenNotMatchedInsert**
#### Dataset ⟶ Dataset of Polish mountaineers

In [ ]:
data_polish = [
    (4, 'Wanda', 'Rutkiewicz', 'Polish', '1944-12-13'),     # This row already exists in the table
    (9, 'Krzysztof', 'Wielicki', 'Polish', '1950-01-05'),   # New row
    (10, 'Lech', 'Cichy', 'Polish', '1951-11-14')            # New row
]
df_polish = session.createDataFrame(data_polish).toDF("id", "first_name", "last_name", "demonym", "date_of_birth").repartition(1)

alpinist.alias("target") \
    .merge(
        df_polish.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsert(
        values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.date_of_birth",
            "nationality": "source.demonym"
        }
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 3 ⟶ Clause **whenMatchedUpdate** and **whenNotMatchedInsert**
#### Dataset ⟶ A dataset of Polish mountaineers already present in the alpinist table, but with some corrections applied

In [ ]:

data_polish_revised = [
    (4, 'Wanda', 'Rutkiewicz', 'Polish', '1943-12-13'),     # Row present in the table with incorrect birthdate | Birthdate correction
    (9, 'Krzysztof', 'Wielicki', 'Polish', '1950-01-05'),   # Row present in the table
    (10, 'Leszek', 'Cichy', 'Polish', '1951-11-14'),         # Row present in the table with incorrect first name | First name correction
    (20, 'Wojciech', 'Kurtyka', 'Polish', '1947-07-25')     # New row
]
df_polish_revised = session.createDataFrame(data_polish_revised).toDF("id", "first_name", "last_name", "demonym", "date_of_birth").repartition(1)

alpinist.alias("target") \
    .merge(
        df_polish_revised.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdate(
        set = {
            "birthdate": "source.date_of_birth",
            "first_name": "source.first_name"
        }
    ) \
    .whenNotMatchedInsert(
        values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.date_of_birth",
            "nationality": "source.demonym"
        }
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 4 ⟶ Clause **whenMatchedUpdate** with **NULL**
#### Dataset ⟶ A dataset of Nepalese mountaineers already present in the alpinist table, but with some corrections applied
The dataset contains only the columns/fields that are modified.

In [ ]:
data_nepalese_revised = [
    (5, None, '1983-07-25'),
    (6, 'Kami', None)
]
df_nepalese_revised = session.createDataFrame(data_nepalese_revised).toDF("id", "first_name", "birthdate").repartition(1)

alpinist.alias("target") \
    .merge(
        df_nepalese_revised.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdate(
        set = {
            "first_name": "CASE WHEN source.first_name IS NOT NULL THEN source.first_name ELSE target.first_name END",
            "birthdate": "CASE WHEN source.birthdate IS NOT NULL THEN source.birthdate ELSE target.birthdate END",
        }
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 5 ⟶ Clause **whenNotMatchedInsert** with **NULL** and **DEFAULT**
#### Dataset ⟶ A dataset of New Zealand mountaineers

In [ ]:
data_newzealander = [
    (11, 'Edmund', 'Hilary', '1919-07-20', 'New Zealander'),
    (12, 'George', 'Lowe', '1924-11-15', None),
    (13, 'Athol', 'Whimp', None, None)
]
df_newzealander = session.createDataFrame(data_newzealander).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_newzealander.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsert(
         values = {
            "id": "source.id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "birthdate": "source.birthdate",
            "nationality": "CASE WHEN source.nationality IS NOT NULL THEN source.nationality ELSE 'New Zealand' END"
        }
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 6 ⟶ Clause **whenMatchedDelete**
#### Dataset ⟶ A dataset of potentially deleted mountaineers
Deleting mountaineers with discrepancies in Birthdate caused by the ambiguous date formats YYYY/MM/DD (US) vs. YYYY/DD/MM (EU).\
Lino Lacedelli: 1925-12-04 vs. 1925-04-12.

In [ ]:
data_deletion = [
    (3, 'Lino', 'Lacedelli', '1925-04-12', 'Italian'),
    (11, 'Edmund', 'Hilary', '1919-07-20', 'New Zealander'),
    (12, 'George', 'Lowe', '1924-11-15', 'New Zealander'),
    (13, 'Athol', 'Whimp', None,  'New Zealander')
]
df_deletion = session.createDataFrame(data_deletion).toDF("id", "first_name", "last_name", "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_deletion.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedDelete(
        condition = "target.birthdate != source.birthdate"
    ) \
    .execute()

alpinist.toDF().show()

### Analysis 7 ⟶ Clause **whenNotMatchedBySourceDelete**
#### Dataset ⟶ A dataset of the first mountaineers, both men and women, to reach the summit of K2


In [ ]:
data = [
    (1, 'Achille', 'Compagnoni', '1914-09-26', 'Italian'),
    (2, 'Ardito', 'Desio', '1897-04-18', 'Italian'),
    (3, 'Lino', 'Lacedelli', "1925-12-04", 'Italian'),
    (4, 'Wanda', 'Rutkiewicz', "1944-12-13", 'Polish')
]
df_pioneer = session.createDataFrame(data).toDF("id", "first_name", "last_name" , "birthdate", "nationality").repartition(1)

alpinist.alias("target") \
    .merge(
        df_pioneer.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedBySourceDelete() \
    .execute()

df_pioneer.show()

## Additions
1. Viewing **alpinist** table data via catalog;
2. Viewing **alpinist** table data via reference.

In [ ]:
session.table("alpinist").show()
alpinist.toDF().show()